# Train Aperture's reference checkpoints

Produces `policy.pt` (**Run A**) and `failure_head.pt` (**Run B**) — the two checkpoints the API
loads when `APERTURE_USE_LEARNED_MODELS=true`.

This notebook is a thin wrapper around [`ml/training/train.py`](../training/train.py). The model
and data code lives in the repo rather than in these cells, for two reasons:

1. **Lockstep is structural.** Training imports `AperturePolicy` / `FailureHead` from
   `aperture.ml.model` — the exact definitions the API loads the checkpoints into. A notebook
   that redefines those classes can silently drift from the API; this one cannot.
2. **It runs anywhere.** Nothing here needs Colab, Google Drive, or the `lerobot` package. The
   same command trains on a laptop (`--device mps` / `cpu`) or a Colab GPU (`--device cuda`).

Runtime: roughly 15 minutes on a Colab T4; a couple of hours on an M-series laptop.

## 1. Repo and dependencies

Skip the clone if you are already running inside a checkout.

In [ ]:
import os
from pathlib import Path

IN_COLAB = Path("/content").exists()
REPO = Path("/content/aperture") if IN_COLAB else Path.cwd().parents[1]

if not REPO.exists():
    !git clone --depth 1 https://github.com/andrewsundaradhas/aperture {REPO}

# The `[ml]` extra pulls torch / timm / sentence-transformers. The two training-only deps are
# the parquet reader and the video decoder.
!pip install -q -e "{REPO}/apps/api[ml]" pyarrow av

os.chdir(REPO / "ml")
print("working in", Path.cwd())

## 2. Train

`--download` fetches `lerobot/xarm_lift_medium` (~18 MB) from the Hub on first run.

| Flag | Meaning |
|---|---|
| `--epochs` | Run A epochs (the policy) |
| `--head-epochs` | Run B epochs — the head trains on cached frozen features, so these are seconds |
| `--failure-samples` | Run B frames per failure surface |
| `--skip-policy` | reuse an existing `policy.pt` and redo Run B only |

Run A writes a checkpoint every epoch to `ml/models/checkpoints/` and resumes from the newest
one, so a dropped Colab session costs at most one epoch.

In [ ]:
!python -m training.train --download --device cuda --epochs 10 --head-epochs 40 --failure-samples 4000

## 3. Inspect what was learned

`metrics.json` carries both runs' history and Run B's held-out report. The validation split is
by **episode**, so no source frame is shared across the split — neighbouring frames in a 15fps
episode are near-duplicates and would otherwise inflate the score.

In [ ]:
import json

metrics = json.loads((Path.cwd() / "models" / "metrics.json").read_text())

print("Run A (policy) — Gaussian NLL")
for row in metrics["run_a_policy"]["history"]:
    print(f"  epoch {row['epoch']:>2}  train {row['train_nll']:.4f}  val {row['val_nll']:.4f}")

report = metrics["run_b_failure_head"]["validation"]
print(f"\nRun B (failure head) — held-out accuracy {report['accuracy']:.3f}")
for surface, scores in report["per_class"].items():
    print(f"  {surface:<11} precision {scores['precision']:.3f}  recall {scores['recall']:.3f}")

print("\nconfusion (rows = true, cols = predicted):", report["confusion_matrix"]["labels"])
for label, row in zip(report["confusion_matrix"]["labels"], report["confusion_matrix"]["rows_true_cols_pred"]):
    print(f"  {label:<11} {row}")

## 4. See the failure-surface data

Each surface is realised as the visual condition that actually distinguishes it — degraded
sensor (perception), several equally plausible referents (grounding), and a clean frame at a
timestep where the logged action trace is anomalous (motor). Worth eyeballing before trusting
anything the head predicts.

In [ ]:
import sys

import numpy as np
from PIL import Image

sys.path.insert(0, str(Path.cwd()))
from training.failure_surfaces import SURFACES, build_failure_dataset
from training.lerobot_data import LeRobotV3Dataset

dataset = LeRobotV3Dataset.load(Path.cwd() / "data" / "xarm_lift_medium")
train_eps, _ = dataset.split_episodes(0.1, seed=0)
images, labels, variants = build_failure_dataset(dataset, train_eps, n_per_class=8, seed=1)

rows = []
for surface_idx, surface in enumerate(SURFACES):
    picks = [i for i, label in enumerate(labels) if label == surface_idx][:8]
    print(f"{surface:<11}", [variants[i] for i in picks])
    rows.append(np.concatenate([images[i] for i in picks], axis=1))

tile = np.concatenate(rows, axis=0)
display(Image.fromarray(tile).resize((tile.shape[1] * 3, tile.shape[0] * 3), Image.NEAREST))

## 5. Test a prediction

On a **real** held-out frame, not random noise — noise tells you the tensor shapes line up and
nothing else.

In [ ]:
import torch
import torchvision.transforms as T

from aperture.ml.model import ACTION_DIM, AperturePolicy, FailureHead

device = "cuda" if torch.cuda.is_available() else "cpu"
models_dir = Path.cwd() / "models"

policy = AperturePolicy(action_dim=ACTION_DIM).to(device).eval()
policy.load_state_dict(torch.load(models_dir / "policy.pt", map_location=device))

head = FailureHead(policy.vision.num_features).to(device).eval()
head.load_state_dict(torch.load(models_dir / "failure_head.pt", map_location=device))

frame = np.asarray(images[0], dtype=np.float32) / 255.0
batch = torch.from_numpy(frame).permute(2, 0, 1).unsqueeze(0).to(device)
batch = T.Resize((224, 224), antialias=True)(batch)
instruction = [dataset.episode_task[int(train_eps[0])]]

with torch.no_grad():
    action_mean, _, attention = policy(batch, instruction)
    probabilities = torch.softmax(head(policy.vision_features(batch)), dim=-1).squeeze(0)

print(f"instruction:    {instruction[0]}")
print(f"true surface:   {SURFACES[labels[0]]}  ({variants[0]})")
print(f"predicted:      {SURFACES[int(probabilities.argmax())]}")
print("probabilities: ", {s: round(float(p), 3) for s, p in zip(SURFACES, probabilities)})
print(f"action mean:    {action_mean.squeeze().cpu().numpy().round(3)}")
print(f"attention:      {tuple(attention.shape)} (language -> patch weights, the heatmap signal)")

## 6. Point the API at the checkpoints

The API reads them straight off disk — no upload step required:

```bash
export APERTURE_USE_LEARNED_MODELS=true
```

`APERTURE_LOCAL_MODEL_DIR` already defaults to the repo's `ml/models/`, so a local checkout
picks these up with just that one variable set.

## 7. (Optional) Publish to the Hugging Face Hub

Only needed for deployments that cannot ship the files alongside the code. Set `REPO_ID` to a
repo **you own**, then set `APERTURE_HF_MODEL_REPO` to the same value.

In [ ]:
from huggingface_hub import HfApi, login

REPO_ID = "your-username/aperture-reference-policy"  # must be a repo you own

login(token=os.environ["HF_TOKEN"])  # in Colab: from google.colab import userdata; userdata.get('HF_TOKEN')
api = HfApi()
api.create_repo(REPO_ID, exist_ok=True)

for name in ("policy.pt", "failure_head.pt", "metrics.json"):
    api.upload_file(path_or_fileobj=str(models_dir / name), path_in_repo=name, repo_id=REPO_ID)
    print("uploaded", name)